# CommGuard adversarial red-team v1

Bounded defensive robustness research. Execution is disabled by default and requires a hash-pinned benign acceptance archive plus explicit approval. The final family/session/config holdout remains sealed unless separately released.


In [ ]:
import importlib
import os
from pathlib import Path
import re
import subprocess
import sys

NOTEBOOK_VERSION = "commguard_adversarial_redteam_v1"
REPOSITORY_URL = "https://github.com/waqasm86/CommGuard.git"
REVIEWED_COMMIT = ""  # Required: immutable 40-character commit visible on origin.
REPOSITORY = Path("/kaggle/working/commguard-source")

if not re.fullmatch(r"[0-9a-f]{40}", REVIEWED_COMMIT):
    raise RuntimeError("Set REVIEWED_COMMIT to the reviewed, pushed 40-character commit SHA.")
if not REPOSITORY.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, str(REPOSITORY)],
        check=True,
    )
if not (REPOSITORY / ".git").is_dir():
    raise RuntimeError(f"Refusing non-Git source directory: {REPOSITORY}")
subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "origin", REVIEWED_COMMIT], check=True)
subprocess.run(
    ["git", "-C", str(REPOSITORY), "checkout", "--detach", REVIEWED_COMMIT], check=True
)
head = subprocess.run(
    ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
dirty = subprocess.run(
    ["git", "-C", str(REPOSITORY), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
pushed_refs = subprocess.run(
    ["git", "-C", str(REPOSITORY), "branch", "-r", "--contains", head],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if head != REVIEWED_COMMIT or dirty or not pushed_refs:
    raise RuntimeError(
        "Reproducibility gate failed: "
        f"head={head} dirty={bool(dirty)} pushed={bool(pushed_refs)}"
    )
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps",
        "-e", str(REPOSITORY),
    ],
    check=True,
)
SOURCE_ROOT = (REPOSITORY / "src").resolve()
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = str(SOURCE_ROOT) + (
    os.pathsep + existing_pythonpath if existing_pythonpath else ""
)
sys.path[:] = [entry for entry in sys.path if Path(entry or ".").resolve() != SOURCE_ROOT]
sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()
for module_name in [
    name for name in sys.modules if name == "commguard" or name.startswith("commguard.")
]:
    del sys.modules[module_name]
import commguard

commguard_path = Path(commguard.__file__).resolve()
try:
    commguard_path.relative_to(SOURCE_ROOT)
except ValueError as exc:
    raise RuntimeError(f"CommGuard imported outside reviewed source: {commguard_path}") from exc
print({
    "reviewed_commit": head,
    "remote_refs": pushed_refs.splitlines(),
    "commguard_import": str(commguard_path.relative_to(REPOSITORY)),
    "torchrun_pythonpath_prefix": os.environ["PYTHONPATH"].split(os.pathsep)[0],
})


In [ ]:
from commguard.artifacts import restore_archive, sha256_file

INPUT_ARCHIVE = Path("/kaggle/input/commguard-detector-evaluation-v2/commguard-detector-evaluation-v2-REPLACE.tar.gz")
EXPECTED_INPUT_SHA256 = ""  # Required: SHA-256 printed by the preceding notebook.
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")

if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_INPUT_SHA256):
    raise RuntimeError("Set EXPECTED_INPUT_SHA256 to the exact 64-character archive hash.")
actual_input_sha256 = sha256_file(INPUT_ARCHIVE)
if actual_input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError(
        "Input archive hash mismatch: "
        f"expected={EXPECTED_INPUT_SHA256} actual={actual_input_sha256}"
    )
restore_archive(INPUT_ARCHIVE, ARTIFACTS, expected_sha256=EXPECTED_INPUT_SHA256)
print({"restored_archive": str(INPUT_ARCHIVE), "sha256": actual_input_sha256})


In [ ]:
from datetime import datetime, timezone

NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")


from commguard.environment.preflight import check_environment, summarize_environment
from commguard.provenance import ProvenanceContext

CONTEXT = ProvenanceContext.create(
    corpus_id=f"corpus-adversarial-v1-{NOTEBOOK_RUN_ID}",
    experiment_session_id=f"session-adversarial-v1-{NOTEBOOK_RUN_ID}",
    collection_id=f"collection-adversarial-v1-{NOTEBOOK_RUN_ID}",
    notebook_version="commguard_adversarial_redteam_v1",
    input_archive_sha256=EXPECTED_INPUT_SHA256,
    random_seed=20260730,
    repository_root=REPOSITORY,
)
if CONTEXT.source_dirty or CONTEXT.source_commit != REVIEWED_COMMIT:
    raise RuntimeError("SDK provenance no longer matches the clean reviewed source commit.")
ENVIRONMENT = check_environment(strict=True, output=ARTIFACTS, provenance=CONTEXT)
print(summarize_environment(ENVIRONMENT))
print({
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "experiment_session_id": CONTEXT.experiment_session_id,
    "collection_id": CONTEXT.collection_id,
    "corpus_id": CONTEXT.corpus_id,
    "source_commit": CONTEXT.source_commit,
    "input_archive_sha256": CONTEXT.input_archive_sha256,
})


In [ ]:
import json

BENIGN_EVALUATION_ARTIFACT_PATH = Path("results/evaluation-REPLACE.json")
if BENIGN_EVALUATION_ARTIFACT_PATH.is_absolute() or ".." in BENIGN_EVALUATION_ARTIFACT_PATH.parts:
    raise RuntimeError("Benign evaluation path must be artifact-root-relative.")
benign_evaluation_path = ARTIFACTS / BENIGN_EVALUATION_ARTIFACT_PATH
if not benign_evaluation_path.is_file():
    raise RuntimeError("Adversarial work requires the exact saved benign evaluation.")
BENIGN_ACCEPTANCE = json.loads(benign_evaluation_path.read_text(encoding="utf-8"))
if not BENIGN_ACCEPTANCE.get("coverage_gate", {}).get("passed"):
    raise RuntimeError("Adversarial work blocked: benign primary coverage did not pass.")
if not BENIGN_ACCEPTANCE.get("primary_communication_only"):
    raise RuntimeError("Adversarial work blocked: benign primary baseline is missing.")
BENIGN_EXTRACTION_SUMMARY = Path(BENIGN_ACCEPTANCE["feature_source"])
if BENIGN_EXTRACTION_SUMMARY.is_absolute():
    matches = sorted((ARTIFACTS / "features").glob(BENIGN_EXTRACTION_SUMMARY.name))
    if len(matches) != 1:
        raise RuntimeError(
            "Cannot resolve the benign extraction summary inside the restored archive."
        )
    BENIGN_EXTRACTION_SUMMARY = matches[0].relative_to(ARTIFACTS)
print({"accepted_benign_evaluation": str(benign_evaluation_path), "coverage": "passed"})


In [ ]:
from commguard.adversarial import ADVERSARIAL_STRATEGIES, AdversarialHoldoutPlan
from commguard.orchestrator import estimate_matrix

HOLDOUT_PLAN = AdversarialHoldoutPlan(
    development_families=(
        "gradient_accumulation",
        "periodic_local_sgd",
        "segmented_runs",
        "idle_padding",
    ),
    hardening_families=(
        "randomized_synchronization",
        "mixed_training_inference",
        "synthetic_communication_decoy",
    ),
    final_family="diloco_inspired",
    final_session_ids=("session-reserved-final-v1",),
    final_config_ids=("diloco-inspired-inner10-v1",),
)
HOLDOUT_PLAN.validate()
for strategy_id, strategy in sorted(ADVERSARIAL_STRATEGIES.items()):
    print({
        "strategy": strategy_id,
        "purpose": strategy.research_purpose,
        "claim_boundary": strategy.claim_boundary,
        "enabled_by_default": strategy.enabled_by_default,
    })
print({"estimate": estimate_matrix("standard", 3), "comparison_only": "benign matrix"})


In [ ]:
from commguard.orchestrator import run_adversarial_matrix

RUN_ADVERSARIAL_PILOT = False
ADVERSARIAL_HUMAN_APPROVAL = False
RELEASE_FINAL_ADVERSARIAL_HOLDOUT = False
FINAL_HOLDOUT_HUMAN_APPROVAL = False

ADVERSARIAL_MATRIX = None
if RUN_ADVERSARIAL_PILOT:
    if not ADVERSARIAL_HUMAN_APPROVAL:
        raise RuntimeError("Set ADVERSARIAL_HUMAN_APPROVAL only after human review.")
    ADVERSARIAL_MATRIX = run_adversarial_matrix(
        output=ARTIFACTS,
        holdout_plan=HOLDOUT_PLAN,
        adversarial_approval=ADVERSARIAL_HUMAN_APPROVAL,
        release_final_adversarial_holdout=RELEASE_FINAL_ADVERSARIAL_HOLDOUT,
        final_holdout_approval=FINAL_HOLDOUT_HUMAN_APPROVAL,
        repetitions=1,
        timeout_s=180.0,
        provenance=CONTEXT,
    )
    for family, row in sorted(ADVERSARIAL_MATRIX["family_counts"].items()):
        print({"family": family, **row})
else:
    print("Adversarial execution remains disabled; no adversarial artifact was created.")


In [ ]:
from commguard.evaluation import evaluate_detector

RUN_ADVERSARIAL_EVALUATION = False
ADVERSARIAL_EVALUATION = None
if RUN_ADVERSARIAL_EVALUATION:
    if ADVERSARIAL_MATRIX is None:
        raise RuntimeError("Collect the approved bounded adversarial matrix first.")
    ADVERSARIAL_EVALUATION = evaluate_detector(
        input_root=ARTIFACTS,
        output=ARTIFACTS,
        benign_extraction_summary=BENIGN_EXTRACTION_SUMMARY,
        adversarial_extraction_summary=ADVERSARIAL_MATRIX["feature_extraction_summary"],
        adversarial_holdout_plan=HOLDOUT_PLAN,
        release_final_adversarial_holdout=RELEASE_FINAL_ADVERSARIAL_HOLDOUT,
    )
    print(ADVERSARIAL_EVALUATION["heldout_adversarial_families"])


## Results

not executed. No adversarial, evasion, decoy, or final-holdout result is claimed.


In [ ]:
if ADVERSARIAL_MATRIX is None:
    raise RuntimeError(
        "NEXT STEP: obtain human approval, set RUN_ADVERSARIAL_PILOT and "
        "ADVERSARIAL_HUMAN_APPROVAL to True, then rerun from a fresh Kaggle session."
    )
from commguard.artifacts import ArtifactStore, sha256_file

ARCHIVE = Path(f"/kaggle/working/commguard-adversarial-redteam-v1-{NOTEBOOK_RUN_ID}.tar.gz")
ArtifactStore(ARTIFACTS).export(ARCHIVE)
ARCHIVE_SHA256 = sha256_file(ARCHIVE)
SHA_FILE = ARCHIVE.with_suffix(ARCHIVE.suffix + ".sha256")
SHA_FILE.write_text(f"{ARCHIVE_SHA256}  {ARCHIVE.name}\n", encoding="utf-8")
print(f"NEXT STEP: add {ARCHIVE} to a private Kaggle dataset without renaming it.")
print(f"NEXT STEP: copy SHA-256 {ARCHIVE_SHA256} into EXPECTED_INPUT_SHA256 in the evidence index and report.")
print(
    f"NEXT STEP: set that notebook's REVIEWED_COMMIT to {REVIEWED_COMMIT} "
    "and run from the first cell."
)
